# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushdevo/10x.ai/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
The baseline action score prioritizes content that already receives meaningful visibility, is ranking near the first page but not at the top, and has not been updated recently.

Reason Codes:

* HIGH_IMPRESSIONS — Strong visibility observed.
* STALE_CONTENT — Content has not been updated recently.
* IMPROVABLE_POSITION — Ranking position suggests optimization opportunity.
* HIGH_PRIORITY_REFRESH — Multiple signals support refreshing.
* LOW_PRIORITY — Limited evidence that refresh action is needed.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("/content/refresh_queue_sample.csv")

scaler = MinMaxScaler()

# Normalize signals
df["imp_norm"] = scaler.fit_transform(df[["impressions_90d"]])

df["stale_norm"] = scaler.fit_transform(
    df[["days_since_last_update"]]
)

df["position_norm"] = scaler.fit_transform(
    df[["avg_position"]]
)

# Baseline score
df["baseline_action_score"] = (
    0.50 * df["imp_norm"] +
    0.30 * df["stale_norm"] +
    0.20 * df["position_norm"]
)

# Rank
df = df.sort_values(
    "baseline_action_score",
    ascending=False
)

df["rank"] = range(1, len(df) + 1)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

df.head(20)

,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,imp_norm,stale_norm,position_norm,baseline_action_score,rank
196,197,content_1bfaa38ff26c,client_7f2253d7e2,76.780950,random_forest,0.721993,0.824268,high,refresh_and_review_engagement,stale_visible_page|declining_with_demand|low_e...,...,181-365,181+,3500+,good,page_3_5,0.563795,1.000000,0.550914,0.692080,1
130,131,content_2333ccd359f7,client_7f2253d7e2,77.339264,random_forest,0.813368,0.642664,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,181-365,0-30,3500+,excellent,page_3_5,0.958496,0.000000,0.527415,0.584731,2
169,170,content_d75823fc91dc,client_7f2253d7e2,77.037774,random_forest,0.830740,0.595457,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,91-180,0-30,3500+,excellent,page_3_5,0.832547,0.000000,0.767624,0.569798,3
31,32,content_964dd0100c99,client_7f2253d7e2,78.945092,random_forest,0.839227,0.636309,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,91-180,0-30,3500+,excellent,page_3_5,0.909015,0.000000,0.550914,0.564690,4
134,135,content_341b1ecdea85,client_7f2253d7e2,77.318512,random_forest,0.820299,0.626925,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,181-365,0-30,3500+,excellent,page_3_5,0.882005,0.000000,0.603133,0.561629,5
99,100,content_9d9905bcb297,client_7f2253d7e2,77.578361,random_forest,0.795638,0.688710,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,91-180,0-30,3500+,excellent,striking,1.000000,0.000000,0.292428,0.558486,6
159,160,content_c9eeaab4031e,client_19581e27de,77.091885,random_forest,0.691605,0.900112,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,91-180,91-180,unknown,excellent,page_1,0.752213,0.482759,0.133159,0.547566,7
106,107,content_ac1d924c6a70,client_7f2253d7e2,77.540917,random_forest,0.750665,0.785475,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,91-180,91-180,2000-3500,good,page_3_5,0.477012,0.494253,0.798956,0.546573,8
145,146,content_c2ac8518bb1f,client_7f2253d7e2,77.234804,random_forest,0.808255,0.650547,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,91-180,0-30,3500+,excellent,striking,0.775808,0.000000,0.454308,0.478766,9
20,21,content_972b37f0c86d,client_19581e27de,79.532099,random_forest,0.748244,0.852687,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,91-180,91-180,3500+,good,page_1,0.602490,0.482759,0.159269,0.477926,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
def reason_code(row):

    reasons = []

    if row["imp_norm"] > 0.7:
        reasons.append("HIGH_IMPRESSIONS")

    if row["stale_norm"] > 0.7:
        reasons.append("STALE_CONTENT")

    if row["position_norm"] > 0.7:
        reasons.append("IMPROVABLE_POSITION")

    if len(reasons) >= 2:
        return "HIGH_PRIORITY_REFRESH"

    if reasons:
        return reasons[0]

    return "LOW_PRIORITY"


top20 = df.head(20).copy()

top20["reason_code"] = top20.apply(
    reason_code,
    axis=1
)

top20[[
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_code"
]]

,rank,content_id,baseline_action_score,reason_code
196,1,content_1bfaa38ff26c,0.692080,STALE_CONTENT
130,2,content_2333ccd359f7,0.584731,HIGH_IMPRESSIONS
169,3,content_d75823fc91dc,0.569798,HIGH_PRIORITY_REFRESH
31,4,content_964dd0100c99,0.564690,HIGH_IMPRESSIONS
134,5,content_341b1ecdea85,0.561629,HIGH_IMPRESSIONS
99,6,content_9d9905bcb297,0.558486,HIGH_IMPRESSIONS
159,7,content_c9eeaab4031e,0.547566,HIGH_IMPRESSIONS
106,8,content_ac1d924c6a70,0.546573,IMPROVABLE_POSITION
145,9,content_c2ac8518bb1f,0.478766,HIGH_IMPRESSIONS
20,10,content_972b37f0c86d,0.477926,LOW_PRIORITY


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
Weak Picks

Some high-ranked items may be misleading because impressions alone can dominate the score. Content with strong visibility but little actual refresh opportunity may appear near the top.

Items with very old content ages may also be prioritized even when rankings are already strong.

Leakage Check

I verified that the baseline score only uses information available at decision time:

* impressions_90d
* avg_position
* days_since_last_update

No future labels, outcomes, model predictions, or refresh results were used.

Specifically, I excluded:

* final_refresh_score
* best_model_probability
* is_declining_label

These fields could leak future information and artificially improve ranking quality.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.